# E15: training-seed variability of the main CNN

**Revision plan, 00 log section 60 (written 23 Oct 2026, before this run).** The main CNN (phase dropout) was chosen among
three candidates whose held-out macro-AUCs differed by at most 0.013 (0.906–0.919), and the CNN was trained once per fold.
This notebook measures how much the results move when only the **training seed** changes. **The main models and all main
results stay unchanged, whatever this shows.**

- **Retrained:** the phase-dropout CNN, E3 protocol on the Week 2 crops, same folds and the same inner validation split; only
  the training seed changes (weight initialization, data order, augmentation and dropout draws), offset per fold by +1000 and
  +2000 (the plan), and +3000, +4000 and +5000 (added before the run, for the standard deviation). Seed 42 is also retrained
  once (offset 0), which measures GPU non-determinism alone.
- **Per seed:** CNN + radiomics and full-fusion meta-learners refitted on that seed's out-of-fold CNN probabilities and the
  saved E2 radiomics probabilities (E8b's meta-learner code).
- **Outcome:** held-out macro-AUC (56) and out-of-fold macro-AUC (222) of the CNN, CNN + radiomics and full fusion, for seed
  42 (the saved E3 models) and each new seed; range and standard deviation.
- **Statement fixed in the plan:** the 0.013 spread of the three CNN candidates is "within" the seed-to-seed variability if it
  is no larger than the range of the CNN's held-out macro-AUC across seeds 42, +1000 and +2000, and "larger than" otherwise.
- **Gates (the notebook stops if one fails):** the saved E3 models reproduce their saved predictions (< 0.02); the meta-learner
  code reproduces the saved E8b fusions from them (< 0.02).

**Runtime:** **A100 GPU** (High-RAM not needed), about 1.5–2 hours (6 trainings of 5 folds). **Resume:** if the run stops, set
`RESUME_RUN_ID` in Cell 1 to the stopped run's time and run all again; trained folds are reused. **When it finishes,** the last
cell saves everything to Drive and disconnects the runtime (`SHUTDOWN_WHEN_DONE = False` keeps it). **If a cell stops with an
error,** the error is saved to Drive (`E15_error.txt`) and the runtime disconnects (`SHUTDOWN_ON_ERROR = False` in Cell 1 keeps
it). **Safety:** inputs are only read; outputs go to a new folder `liver_project/revision_2026-10/E15_<time>/`.

In [ ]:
# Cell 1 — Setup (E10's inputs, no radiomics): installs, paths, input checks, output folder E15_<time>, resume, disconnect on error
import os, sys, json, time, hashlib, platform, datetime, warnings, subprocess, random
from pathlib import Path
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(os.environ['COLAB_DRIVE_MOUNT'])  # see docs/revision_notebooks.md
    ROOT = Path(os.environ['LIVER_PROJECT_ROOT'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'monai'], check=True)
else:
    ROOT = Path(os.environ['E15_TEST_ROOT'])

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from monai.networks.nets import resnet18
from scipy import ndimage
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

PROJECT_DIR = ROOT / 'liver_project'
REVISION_DIR = PROJECT_DIR / 'revision_2026-10'
E2_DIR = REVISION_DIR / 'E1_E2_20261017T032941Z'
E3_DIR = REVISION_DIR / 'E3_20261017T041700Z'
E8B_DIR = REVISION_DIR / 'E8b_20261018T025705Z'
SEED = 42
PHASE_ORDER = ['P', 'C1', 'C2', 'C3']
CLASSES = ['HCC', 'ICC', 'CHCC']
P3 = ['p_HCC', 'p_ICC', 'p_CHCC']
# fixed in the plan (00 log section 60): +1000 and +2000; +3000 to +5000 and the rerun of seed 42 (offset 0) added before the run
SEED_OFFSETS = [int(x) for x in os.environ.get('E15_TEST_OFFSETS', '0,1000,2000,3000,4000,5000').split(',')]
PLAN_OFFSETS = [1000, 2000]
CANDIDATE_SPREAD = 0.013                      # Section 4.2: the three CNN candidates, held-out macro-AUC 0.906-0.919
INPUTS = {
    'split': (PROJECT_DIR / 'splits' / 'train_val_test.json', 'cfbe3cb6f9b1901c3d672042e7918a95051d841d328d8353492f91376617513e'),
    'radiomics_features': (PROJECT_DIR / 'radiomics' / 'radiomics_features.parquet', 'caac3d2cf4f273bd6eebd48154a1dc4bf20adbbb7a896b3c4947a910fd01e156'),
    'e2_oof_meta': (E2_DIR / 'E2_fusion_oof_meta_features_noshape.csv', 'c20d23d3d0a7df871212f6f937bc2024e686c11dc6975350329cbe186f058d12'),
    'e2_test_meta': (E2_DIR / 'E2_fusion_test_meta_features_noshape.csv', 'fbb946f641a050f66b093564141a8e103e1c4e5c651d6abeb51042293ab8cb2f'),
    'e2_radiomics_test': (E2_DIR / 'E2_radiomics_noshape_test_predictions.csv', '9aa460d2ebe597b8b116006af9895fbf9f6fe55f87a9268bef69d69f2830db90'),
    'e3_d_predictions': (E3_DIR / 'E3B_D_phase_dropout_predictions.csv', '4c1aed86c7e8095390b8c4313cb69e12c31bd87a6e67a518e23e3e3e95470ee6'),
    'e8b_predictions': (E8B_DIR / 'E8b_E9_predictions.csv', '644a520c0d034ce233bb88ab98e1b6d9e89993a661e380d06985edde4aa34064'),
}
for f in range(5):
    INPUTS[f'fold_{f}'] = (PROJECT_DIR / 'splits' / f'fold_{f}.json', None)
MODEL_FILES = {f'd_{f}': E3_DIR / 'D_phase_dropout' / f'fold_{f}.pt' for f in range(5)}
def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()
problems, input_hashes = [], {}
for key, (path, expected) in INPUTS.items():
    if not path.exists():
        problems.append(f'missing: {path}'); continue
    input_hashes[key] = sha256_of(path)
    if expected and input_hashes[key] != expected:
        (problems if IN_COLAB else []).append(f'changed: {path.name}')
problems += [f'missing: {p}' for p in MODEL_FILES.values() if not p.exists()]
tar_path = PROJECT_DIR / 'processed.tar'
if not tar_path.exists() or (IN_COLAB and tar_path.stat().st_size != 402616320):
    problems.append(f'missing or changed size: {tar_path}')
if problems:
    raise RuntimeError('STOP — input check failed:\n' + '\n'.join(problems))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import sklearn, monai
VERSIONS = {'python': platform.python_version(), 'torch': torch.__version__, 'monai': monai.__version__, 'scikit-learn': sklearn.__version__,
            'numpy': np.__version__, 'device': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'cpu', 'cpus': os.cpu_count()}
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RESUME_RUN_ID = None   # to continue a stopped run, put its time here, e.g. '20261024T020304Z'
if RESUME_RUN_ID:
    RUN_ID = RESUME_RUN_ID
OUT_DIR = REVISION_DIR / f'E15_{RUN_ID}'
OUT_DIR.mkdir(parents=False, exist_ok=bool(RESUME_RUN_ID))

SHUTDOWN_ON_ERROR = True   # if any cell stops with an error: save it to Drive (E15_error.txt) and disconnect the runtime
if IN_COLAB and SHUTDOWN_ON_ERROR:
    from IPython import get_ipython
    def _disconnect_on_error(shell, etype, evalue, tb, tb_offset=None):
        shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
        try:
            (OUT_DIR / 'E15_error.txt').write_text(f'{etype.__name__}: {evalue}')
            from google.colab import drive as _drive, runtime as _runtime
            print('\nThe run stopped with the error above; it is saved in Drive (E15_error.txt). The runtime disconnects '
                  'in 60 seconds to save compute units; this output stays visible. Send it for review.')
            _drive.flush_and_unmount()
            time.sleep(60)
            _runtime.unassign()
        except Exception as error:
            print('Could not disconnect automatically:', error)
    get_ipython().set_custom_exc((Exception,), _disconnect_on_error)
print('Inputs and models checked:', len(INPUTS) + len(MODEL_FILES), '| versions:', VERSIONS, '| seed offsets:', SEED_OFFSETS,
      '| output folder:', OUT_DIR.name)

In [ ]:
# Cell 2 — Data: split, folds, labels, the Week 2 processed crops (the E3 training inputs)
if IN_COLAB:
    PROCESSED_DIR = Path('/content/processed')
    if len(list(PROCESSED_DIR.glob('*.npz'))) < 278:
        subprocess.run(['cp', str(tar_path), '/content/processed.tar'], check=True)
        subprocess.run(['tar', '-xf', '/content/processed.tar', '-C', '/content/'], check=True)
        os.remove('/content/processed.tar')
else:
    PROCESSED_DIR = Path(os.environ['E15_TEST_PROCESSED'])
split = json.loads(INPUTS['split'][0].read_text())
dev_ids, test_ids = [str(x) for x in split['dev_ids']], [str(x) for x in split['test_ids']]
ALL_IDS = dev_ids + test_ids
fold_splits = [json.loads(INPUTS[f'fold_{f}'][0].read_text()) for f in range(5)]
FOLD_OF = {str(p): f for f, s in enumerate(fold_splits) for p in s['val_ids']}
assert set(FOLD_OF) == set(dev_ids), 'fold files do not cover the development set exactly'
internal = pd.read_parquet(INPUTS['radiomics_features'][0])
internal['patient_id'] = internal['patient_id'].astype(str)
LABEL = internal.set_index('patient_id')['label'].astype(int).to_dict()
CLS = {p: CLASSES[LABEL[p]] for p in ALL_IDS}
CT = {}
for pid in ALL_IDS:
    with np.load(PROCESSED_DIR / f'{pid}.npz') as z:
        CT[pid] = z['ct'].astype(np.uint8)
assert all(v.shape == (4, 96, 96, 96) for v in CT.values())
print('Patients:', len(ALL_IDS), '| classes', pd.Series(CLS).value_counts().to_dict(), '| crops in memory:', len(CT))

In [ ]:
# Cell 3 — Training code (E8b's E3 protocol; E15 changes only the training seed); gate 1 with the saved E3 models
NUM_EPOCHS = int(os.environ.get('E15_TEST_EPOCHS', 40))
BATCH_SIZE, LR, WD = 4, 1e-4, 1e-5
USE_AMP = DEVICE.type == 'cuda'
WORKERS = 4 if IN_COLAB else 0
SEED_OFFSET = 0

# Cell 6 — Training code: E3 phase-dropout protocol, plus scanner-variation augmentation and the no-non-contrast option
def zscore4(x4):                              # Week 4 / E3, unchanged
    out = np.zeros_like(x4, dtype=np.float32)
    for c in range(x4.shape[0]):
        vol = x4[c]
        mask = vol > 0
        if mask.sum() < 100:
            out[c] = vol.astype(np.float32)
            continue
        vals = vol[mask]
        out[c][mask] = (vals - vals.mean()) / (vals.std() + 1e-6)
    return out

def augment(vol4):                            # Week 4 / E3, unchanged
    if random.random() < 0.5:
        vol4 = vol4[:, ::-1].copy()
    if random.random() < 0.5:
        vol4 = vol4[:, :, ::-1].copy()
    if random.random() < 0.5:
        for c in range(vol4.shape[0]):
            scale = 1.0 + (random.random() - 0.5) * 0.2
            shift = (random.random() - 0.5) * 0.2
            mask = vol4[c] != 0
            vol4[c][mask] = vol4[c][mask] * scale + shift
    return vol4

def sample_kept(allowed):
    """E3 phase dropout over the allowed channels: all allowed with p = 0.5, else drop each with p = 0.5, keep >= 1."""
    if random.random() < 0.5:
        return tuple(allowed)
    while True:
        kept = tuple(c for c in allowed if random.random() >= 0.5)
        if kept:
            return kept

def keep_only(vol4, kept):
    out = vol4.copy()
    for c in range(4):
        if c not in kept:
            out[c] = 0.0
    return out

def domain_randomise(raw4):
    """Scanner-variation augmentation (plan 21, E8b-2) on the 0-255 crop before normalisation; one draw for all phases."""
    x = raw4.astype(np.float32)
    mask = x > 0
    if random.random() < 0.5:
        sigma = random.uniform(0.0, 1.5)
        x = np.stack([ndimage.gaussian_filter(x[c], sigma) for c in range(4)])
    if random.random() < 0.5:
        x = x + np.random.normal(0.0, random.uniform(0.0, 10.0), x.shape).astype(np.float32)
    if random.random() < 0.5:
        gamma = random.uniform(0.7, 1.5)
        x = 255.0 * (np.clip(x, 0, 255) / 255.0) ** gamma
    if random.random() < 0.5:
        f = random.uniform(0.5, 1.0)
        down = [ndimage.zoom(x[c], (f, f, 1.0), order=1) for c in range(4)]
        x = np.stack([ndimage.zoom(d, (96 / d.shape[0], 96 / d.shape[1], 1.0), order=1)[:96, :96, :96] for d in down])
    if random.random() < 0.5:
        sk = random.uniform(0.0, 2.0)
        x = np.stack([ndimage.gaussian_filter(x[c], (0, 0, sk)) for c in range(4)])
    if random.random() < 0.5:
        x = x * random.uniform(0.9, 1.1)
    return (np.clip(x, 0, 255) * mask).astype(np.float32)

class CropDataset(Dataset):
    def __init__(self, ids, arrays, train=False, phase_dropout=False, dr=False, allowed=(0, 1, 2, 3), kept=None):
        self.ids, self.arrays, self.train, self.phase_dropout, self.dr = list(ids), arrays, train, phase_dropout, dr
        self.allowed, self.kept = tuple(allowed), tuple(kept) if kept is not None else tuple(allowed)
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, index):
        pid = self.ids[index]
        raw = self.arrays[pid].astype(np.float32)
        if self.train and self.dr:
            raw = domain_randomise(raw)
        vol4 = zscore4(raw)
        if self.train:
            vol4 = augment(vol4)
        kept = sample_kept(self.allowed) if (self.train and self.phase_dropout) else self.kept
        if kept != (0, 1, 2, 3):
            vol4 = keep_only(vol4, kept)
        return torch.from_numpy(np.ascontiguousarray(vol4, dtype=np.float32)), LABEL.get(pid, 0), pid

def seed_worker(worker_id):                  # each loader worker gets its own numpy stream (torch seeds python/torch per worker)
    np.random.seed(torch.initial_seed() % 2 ** 32)

def make_model():
    return resnet18(spatial_dims=3, n_input_channels=4, num_classes=3, feed_forward=True)

def class_weights_for(ids):
    counts = np.bincount([LABEL[p] for p in ids], minlength=3).astype(np.float32)
    return torch.from_numpy(counts.sum() / (3 * counts.clip(min=1))).float()

def macro_auc(y, p):
    return float(roc_auc_score(np.asarray(y), np.asarray(p), multi_class='ovr', average='macro'))

@torch.no_grad()
def predict(model, ids, arrays, kept=(0, 1, 2, 3), batch_size=8):
    model.eval()
    loader = DataLoader(CropDataset(ids, arrays, kept=kept), batch_size=batch_size, shuffle=False,
                        num_workers=2 if IN_COLAB else 0, pin_memory=USE_AMP)
    probs, pids = [], []
    for x, _, batch_ids in loader:
        probs.append(torch.softmax(model(x.to(DEVICE, non_blocking=True)).float().cpu(), dim=1).numpy())
        pids.extend(batch_ids)
    return pd.DataFrame(np.concatenate(probs), index=pids, columns=P3)

def set_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if USE_AMP:
        torch.cuda.manual_seed_all(seed)

def train_fold(arm, cfg, fold):
    path = OUT_DIR / arm / f'fold_{fold}.pt'
    if path.exists():
        print(f'  {arm} fold {fold}: already trained, loading.')
        return
    path.parent.mkdir(exist_ok=True)
    outer_train = list(fold_splits[fold]['train_ids'])
    inner_train, inner_val = train_test_split(outer_train, test_size=0.2, random_state=SEED + fold,
                                              stratify=[LABEL[p] for p in outer_train])
    set_seeds(SEED + SEED_OFFSET + fold)            # E15: only the training seed changes; the inner split stays E3's
    loader = DataLoader(CropDataset(inner_train, CT, train=True, phase_dropout=True, dr=cfg['dr'], allowed=cfg['allowed']),
                        batch_size=BATCH_SIZE, shuffle=True, num_workers=WORKERS, pin_memory=USE_AMP, drop_last=True,
                        persistent_workers=WORKERS > 0, worker_init_fn=seed_worker)
    model = make_model().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    criterion = nn.CrossEntropyLoss(weight=class_weights_for(inner_train).to(DEVICE))
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    y_inner = np.array([LABEL[p] for p in inner_val])
    best_auc, best_state, best_epoch, history = -1.0, None, -1, []
    for epoch in range(NUM_EPOCHS):
        model.train()
        started, loss_sum, n = time.time(), 0.0, 0
        for x, y, _ in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * x.size(0)
            n += x.size(0)
        scheduler.step()
        inner_auc = macro_auc(y_inner, predict(model, inner_val, CT, kept=cfg['allowed']).loc[inner_val].to_numpy())
        history.append({'epoch': epoch, 'train_loss': loss_sum / max(n, 1), 'inner_val_auc': inner_auc, 'seconds': time.time() - started})
        if inner_auc > best_auc:
            best_auc, best_epoch = inner_auc, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    torch.save({'state_dict': best_state, 'arm': arm, 'fold': fold, 'best_epoch': best_epoch, 'best_inner_val_auc': best_auc,
                'config': {k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()}, 'epochs': NUM_EPOCHS}, path)
    pd.DataFrame(history).to_csv(OUT_DIR / arm / f'fold_{fold}_history.csv', index=False)
    print(f'  {arm} fold {fold}: best inner-val AUC {best_auc:.4f} at epoch {best_epoch} ({sum(h["seconds"] for h in history) / 60:.1f} min)', flush=True)
    del model
    if USE_AMP:
        torch.cuda.empty_cache()

def load_state(path):
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    model = make_model().to(DEVICE)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    return model


def cnn_probs(models):                        # E10: development patients with their own fold model, held-out with the 5-model ensemble
    oof = pd.concat([predict(models[f], [str(p) for p in fold_splits[f]['val_ids']], CT) for f in range(5)]).loc[dev_ids]
    test = sum(predict(m, test_ids, CT).loc[test_ids] for m in models) / len(models)
    return oof, test

SAVED_MODELS = [load_state(MODEL_FILES[f'd_{f}']) for f in range(5)]
SAVED_OOF, SAVED_TEST = cnn_probs(SAVED_MODELS)
e3 = pd.read_csv(INPUTS['e3_d_predictions'][0], dtype={'patient_id': str})
e3 = e3[e3['pattern'] == 'complete']
GATE = {'cnn_saved_vs_E3': float(max(np.abs(SAVED_OOF.to_numpy() - e3[e3['split'] == 'oof'].set_index('patient_id').loc[dev_ids, P3].to_numpy()).max(),
                                     np.abs(SAVED_TEST.to_numpy() - e3[e3['split'] == 'test'].set_index('patient_id').loc[test_ids, P3].to_numpy()).max()))}
print('Gate 1 — saved E3 models vs their saved predictions (max difference, must be < 0.02):', GATE['cnn_saved_vs_E3'])
if IN_COLAB and GATE['cnn_saved_vs_E3'] > 0.02:
    raise RuntimeError('STOP — the saved E3 models do not reproduce their predictions. Send this output.')

In [ ]:
# Cell 4 — Train the phase-dropout CNN once per seed offset (5 folds each; trained folds are reused on resume)
ARM_CFG = {'dr': False, 'allowed': (0, 1, 2, 3)}          # E3's phase dropout (as E11/E12)
arm_of = lambda off: f'phase_dropout_seed_offset_{off}'
t0 = time.time()
for SEED_OFFSET in SEED_OFFSETS:
    print(f'Seed offset +{SEED_OFFSET} (training seeds {SEED + SEED_OFFSET}-{SEED + SEED_OFFSET + 4})', flush=True)
    for fold in range(5):
        train_fold(arm_of(SEED_OFFSET), ARM_CFG, fold)
    print(f'  done ({(time.time() - t0) / 60:.1f} min so far)', flush=True)

In [ ]:
# Cell 5 — Predictions per seed, meta-learners refitted per seed (E8b's code), gate 2, results and the pre-specified statement
W4, W5 = ['w4_p_HCC', 'w4_p_ICC', 'w4_p_CHCC'], ['w5_p_HCC', 'w5_p_ICC', 'w5_p_CHCC']
FUSIONS = {'CNN + radiomics': W4 + W5, 'full fusion': W4 + W5 + ['age', 'gender']}
oof_meta0 = pd.read_csv(INPUTS['e2_oof_meta'][0], dtype={'patient_id': str}).set_index('patient_id').loc[dev_ids].reset_index()
test_meta = pd.read_csv(INPUTS['e2_test_meta'][0], dtype={'patient_id': str}).set_index('patient_id').loc[test_ids]
RAD_TEST = pd.read_csv(INPUTS['e2_radiomics_test'][0], dtype={'patient_id': str}).set_index('patient_id').loc[test_ids, P3]
def prepare(X):                               # E8b, unchanged
    X = X.astype(np.float32)
    X[X <= -1] = np.nan
    return X
def fused(cnn_oof, cnn_test):                 # E8b meta_folds, unchanged; applied as E10 (own fold for development, mean for held-out)
    oof_meta = oof_meta0.copy()
    oof_meta[W4] = cnn_oof.loc[oof_meta['patient_id']].to_numpy()
    out = {}
    for name, cols in FUSIONS.items():
        fitted = []
        for f in range(5):
            tr = oof_meta['fold'].astype(int).ne(f).to_numpy()
            X = prepare(oof_meta.loc[tr, cols].to_numpy())
            imp = SimpleImputer(strategy='median').fit(X)
            sc = StandardScaler().fit(imp.transform(X))
            m = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED).fit(sc.transform(imp.transform(X)), oof_meta.loc[tr, 'label'].astype(int))
            fitted.append((imp, sc, m))
        apply = lambda fits, X: np.mean([m.predict_proba(s.transform(i.transform(prepare(X)))) for i, s, m in fits], axis=0)
        X_dev = oof_meta.set_index('patient_id')[cols]
        oof = pd.DataFrame([apply([fitted[FOLD_OF[p]]], X_dev.loc[[p]].to_numpy())[0] for p in dev_ids], index=dev_ids, columns=P3)
        X_test = np.column_stack([cnn_test.loc[test_ids].to_numpy(), RAD_TEST.loc[test_ids].to_numpy()]
                                 + ([test_meta[['age', 'gender']].to_numpy(dtype=np.float64)] if name == 'full fusion' else []))
        out[name] = (oof, pd.DataFrame(apply(fitted, X_test), index=test_ids, columns=P3))
    return out

RUNS = {'seed 42 (saved E3)': (SAVED_OOF, SAVED_TEST)}
for off in SEED_OFFSETS:
    models = [load_state(OUT_DIR / arm_of(off) / f'fold_{f}.pt') for f in range(5)]
    RUNS['seed 42 (retrained)' if off == 0 else f'seed 42 + {off}'] = cnn_probs(models)
    del models
PRED_ROWS, FUSED = [], {}
for run, (c_oof, c_test) in RUNS.items():
    FUSED[run] = {'CNN': (c_oof, c_test), **fused(c_oof, c_test)}
    for model, (oof, test) in FUSED[run].items():
        PRED_ROWS += [{'run': run, 'model': model, 'cohort': cohort, 'patient_id': p, **frame.loc[p, P3].to_dict()}
                      for cohort, frame in [('internal_oof', oof), ('internal_test', test)] for p in frame.index]
pd.DataFrame(PRED_ROWS).to_csv(OUT_DIR / 'E15_predictions.csv', index=False)

# gate 2: from the saved E3 models, the meta-learner code reproduces the saved E8b fusions
e8b = pd.read_csv(INPUTS['e8b_predictions'][0], dtype={'patient_id': str})
E8B_KEYS = {'CNN + radiomics': 'cnn_plus_radiomics: [ref] phase_dropout_CNN + radiomics_noshape', 'full fusion': 'full_fusion: [ref] phase_dropout_CNN + radiomics_noshape'}
diffs = []
for model, key in E8B_KEYS.items():
    t = e8b[e8b['model'] == key]
    oof, test = FUSED['seed 42 (saved E3)'][model]
    diffs += [np.abs(oof.to_numpy() - t[t['cohort'] == 'internal_oof'].set_index('patient_id').loc[dev_ids, P3].to_numpy()).max(),
              np.abs(test.to_numpy() - t[t['cohort'] == 'internal_test'].set_index('patient_id').loc[test_ids, P3].to_numpy()).max()]
GATE['fusions_saved_vs_E8b'] = float(max(diffs))
print('Gate 2 — meta-learners on the saved E3 models vs the saved E8b fusions (max difference, must be < 0.02):', GATE['fusions_saved_vs_E8b'])
if IN_COLAB and GATE['fusions_saved_vs_E8b'] > 0.02:
    raise RuntimeError('STOP — the meta-learner code does not reproduce E8b. Send this output.')

def macro_auc(y, p):
    return float(roc_auc_score(np.asarray(y), np.asarray(p), multi_class='ovr', average='macro'))
y_dev, y_test = np.array([LABEL[p] for p in dev_ids]), np.array([LABEL[p] for p in test_ids])
two = np.isin(y_test, [0, 1])
rows = []
for run, models in FUSED.items():
    for model, (oof, test) in models.items():
        p = test.loc[test_ids].to_numpy(); pred = p.argmax(1)
        rows.append({'run': run, 'model': model, 'heldout_macro_auc': macro_auc(y_test, p), 'oof_macro_auc': macro_auc(y_dev, oof.loc[dev_ids].to_numpy()),
                     'heldout_hcc_vs_icc_auc': float(roc_auc_score(y_test[two] == 0, p[two, 0] / np.clip(p[two, 0] + p[two, 1], 1e-12, None))),
                     'heldout_accuracy': float((pred == y_test).mean()),
                     **{f'recall_{c}': float(((pred == i) & (y_test == i)).sum() / (y_test == i).sum()) for i, c in enumerate(CLASSES)}})
RESULTS = pd.DataFrame(rows)
PLAN_RUNS = ['seed 42 (saved E3)'] + [f'seed 42 + {o}' for o in PLAN_OFFSETS]
SEED_RUNS = ['seed 42 (saved E3)'] + [f'seed 42 + {o}' for o in SEED_OFFSETS if o != 0]
spread = []
for model in FUSED['seed 42 (saved E3)']:
    for metric in ['heldout_macro_auc', 'oof_macro_auc', 'heldout_hcc_vs_icc_auc', 'heldout_accuracy']:
        r = RESULTS[RESULTS['model'] == model].set_index('run')[metric]
        plan_vals, all_vals = r.reindex(PLAN_RUNS).dropna(), r.reindex(SEED_RUNS).dropna()
        spread.append({'model': model, 'metric': metric, 'plan_seeds': len(plan_vals), 'range_plan_seeds': float(plan_vals.max() - plan_vals.min()),
                       'all_seeds': len(all_vals), 'mean_all_seeds': float(all_vals.mean()), 'sd_all_seeds': float(all_vals.std(ddof=1)) if len(all_vals) > 1 else np.nan,
                       'min_all_seeds': float(all_vals.min()), 'max_all_seeds': float(all_vals.max()),
                       'retrained_seed_42_minus_saved': float(r.get('seed 42 (retrained)', np.nan) - r['seed 42 (saved E3)'])})
SPREAD = pd.DataFrame(spread)
cnn_range = SPREAD.set_index(['model', 'metric']).loc[('CNN', 'heldout_macro_auc'), 'range_plan_seeds']
PLAN_COMPLETE = all(run in RUNS for run in PLAN_RUNS)
STATEMENT = (('the 0.013 spread of the three CNN candidates is within the seed-to-seed variability' if CANDIDATE_SPREAD <= cnn_range
              else 'the 0.013 spread of the three CNN candidates is larger than the seed-to-seed variability')
             + f' (range of the CNN held-out macro-AUC across seeds 42, +1000 and +2000: {cnn_range:.3f})') if PLAN_COMPLETE else 'plan seeds not all trained'
RESULTS.to_csv(OUT_DIR / 'E15_results_per_seed.csv', index=False)
SPREAD.to_csv(OUT_DIR / 'E15_seed_spread.csv', index=False)
histories = []
for off in SEED_OFFSETS:
    for f in range(5):
        hp = OUT_DIR / arm_of(off) / f'fold_{f}_history.csv'
        ck = torch.load(OUT_DIR / arm_of(off) / f'fold_{f}.pt', map_location='cpu', weights_only=False)
        histories.append({'seed_offset': off, 'fold': f, 'best_epoch': ck['best_epoch'], 'best_inner_val_auc': ck['best_inner_val_auc'],
                          'minutes': float(pd.read_csv(hp)['seconds'].sum() / 60) if hp.exists() else np.nan})
TRAINING = pd.DataFrame(histories)
TRAINING.to_csv(OUT_DIR / 'E15_training_per_fold.csv', index=False)
summary = {'run_id': RUN_ID, 'notebook': 'R19_E15_cnn_training_seed_variability.ipynb', 'versions': VERSIONS, 'input_sha256': input_hashes, 'gates': GATE, 'epochs': NUM_EPOCHS,
           'seed_offsets': SEED_OFFSETS, 'candidate_spread': CANDIDATE_SPREAD, 'statement': STATEMENT,
           'protocol': '00 log section 60, E15 (written 23 Oct 2026, before the run); the main models are unchanged'}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))
manifest = [{'file': p.relative_to(OUT_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of(p)} for p in sorted(OUT_DIR.rglob('*')) if p.is_file()]
pd.DataFrame(manifest).to_csv(OUT_DIR / 'output_manifest.csv', index=False)

print('\n================ COPY FROM HERE AND SEND BACK ================')
print(f'Run E15_{RUN_ID} | epochs {NUM_EPOCHS} | seed offsets {SEED_OFFSETS} | gates: {GATE}')
print('\n--- results per seed (held-out 56, out-of-fold 222) ---'); print(RESULTS.round(4).to_string(index=False))
print('\n--- spread across seeds ---'); print(SPREAD.round(4).to_string(index=False))
print('\n--- training per fold ---'); print(TRAINING.round(4).to_string(index=False))
print('\nPre-specified statement:', STATEMENT)
print('==============================================================')

In [ ]:
# Cell 6 — Finish: make sure every result file has reached Google Drive, then disconnect the runtime (stops compute units)
SHUTDOWN_WHEN_DONE = True     # set to False to keep the runtime after the run
if IN_COLAB and SHUTDOWN_WHEN_DONE:
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    print('All files are saved in Drive. The runtime disconnects in 30 seconds; the outputs above stay visible on this page.')
    time.sleep(30)
    runtime.unassign()